<a href="https://colab.research.google.com/github/YBcho02/gasoline-price-search-index/blob/main/notebook/Data_Collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setting

In [6]:
# !pip install trendspy
# USE Trenspy libaray : https://github.com/sdil87/trendspy

In [1]:
import json
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry
from requests import status_codes
#from pytrends import exceptions
from urllib.parse import quote
import datetime
from datetime import datetime, date, time
import time
import matplotlib.pyplot as plt
import numpy as np
import random
from trendspy import Trends

In [5]:

RAW  = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
RAW.mkdir(parents=True, exist_ok=True)
PROC.mkdir(parents=True, exist_ok=True)

## Keywordlist(27 Aug, 2026)

we began with 8 ground keywords from top 25 related queries of the keyword "Gasoline" which is placed under the named category "Fuel" by Google itself: *gas, gas station, gas prices, gas mileage, gas tank, gasoline, gas stations* and *gas price*. Two related queries from top 10 are removed because they include 'near me' in themselves, which is the byproduct of Google search's location service launched in 2015.

Duplicates and irrelevant queries such as ’shell’(too general
and broad), ’natural gas’, ’air gas’(not about gasoline), ’toyota’(too broad car name) are removed.

In [2]:

REPO = "gasoline-price-search-index"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    ROOT = Path("/content") / REPO
    if ROOT.exists():
        !git -C {ROOT} pull -q
    else:
        !git clone -q https://github.com/YBcho02/{REPO}.git {ROOT}
else:
    ROOT = Path.cwd().parent

sys.path.append(str(ROOT / "src"))
from keywords import KEYWORDS, ANCHOR
print(f"{len(KEYWORDS)} keywords, anchor = '{ANCHOR}'")

61 keywords, anchor = 'gas price'


## Retrieve a single Search Volume Index
using Trendspy package, We download monthly data of search requests made in the U.S. from January 2010 to August 2026.

In [4]:


tr = Trends(request_delay=3.0)
series = {}
failed = []

for word in KEYWORDS:
    for attempt in range(1, 6):
        try:
            df = tr.interest_over_time(
                [word], timeframe='2010-01-01 2026-08-01', geo='US'
            )
            if df is None or df.empty:
                failed.append(word)
                print(f"Empty result for '{word}'.")
            else:
                col = word if word in df.columns else df.columns[0]
                series[word] = df[col].rename(word)
                print(f"Success for '{word}'.")
            break
        except Exception as e:
            print(f"'{word}' attempt {attempt}: {type(e).__name__}: {e}")
            time.sleep(30 * attempt)
    else:
        failed.append(word)
        print(f"CRITICAL: gave up on '{word}'.")

    time.sleep(random.uniform(5, 10))

Wdata = pd.concat(series.values(), axis=1) if series else pd.DataFrame()

print(f"Done. {len(series)} retrieved, {len(failed)} failed: {failed}")

Success for 'average gas prices'.
Success for 'average price of gas'.
Success for 'best gas mileage'.
Success for 'best gas mileage car'.
Success for 'best gas mileage cars'.
Success for 'best gas prices'.
Success for 'best suv gas mileage'.
Success for 'ca gas prices'.
Success for 'california gas price'.
Success for 'california gas prices'.
Success for 'car gas mileage'.
Success for 'cheap gas'.
Success for 'cheap gas prices'.
Success for 'costco gas'.
Success for 'costco gasoline'.
Success for 'current gas price'.
Success for 'gallon of gas price'.
Success for 'gas buddy'.
Success for 'gas calculator'.
Success for 'gas costco price'.
Success for 'gas mileage'.
Success for 'gas mileage calculator'.
Success for 'gas mileage for cars'.
Success for 'gas price'.
Success for 'gas price average'.
Success for 'gas price news'.
Success for 'gas price per gallon'.
Success for 'gas price today'.
Success for 'gas prices'.
Success for 'gas prices costco'.
Success for 'gas prices in california'.
S

In [7]:

Wdata.to_csv(RAW / "gtrends_single_terms.csv")                          # save -> download -> uploaded to github


# Retrieving relative batch

We re-scale all the SVI series to make them comparable across each other, using “gas
price” as the key base term. To do so, we need to retrieve a pair of keywords together and compute a ratio : search term popularity to anchor term popularity.

In [9]:


ANCHOR = 'gas price'
TIMEFRAME = '2010-01-01 2026-08-01'

tr = Trends(request_delay = 3.0)
raw = {}
failed = []

for word in KEYWORDS:
    if word == ANCHOR:
        continue   # a duplicated keyword in one payload breaks the request

    for attempt in range(1, 6):
        try:
            df = tr.interest_over_time([ANCHOR, word], timeframe=TIMEFRAME, geo='US')

            if df is None or df.empty or ANCHOR not in df.columns or word not in df.columns:
                failed.append(word)
                print(f"Empty or malformed result for '{word}'.")
            else:
                raw[word] = df[[ANCHOR, word]].astype(float)
                print(f"Success for '{word}'.")
            break

        except Exception as e:
            print(f"'{word}' attempt {attempt}: {type(e).__name__}: {e}")
            time.sleep(30 * attempt)
    else:
        failed.append(word)
        print(f"CRITICAL: gave up on '{word}'.")

    time.sleep(random.uniform(5, 10))

# --- rescaling ---
ratio, scaled, diag = {}, {}, []

for word, df in raw.items():
    a, w = df[ANCHOR], df[word]
    a_mean = a.mean()

    diag.append({
        'word': word,
        'anchor_mean': a_mean,
        'anchor_zero_share': (a == 0).mean(),
        'word_mean': w.mean(),
    })

    # pointwise ratio # 0/0 and x/0 turned into NaN
    ratio[word] = (w / a.replace(0, np.nan)).rename(word)

    # level-only rescaling: keeps the word's own time shape
    scaled[word] = (w / a_mean if a_mean > 0
                    else pd.Series(np.nan, index=w.index)).rename(word)

Wdata_Wgs_weight_only = pd.concat(ratio.values(), axis=1) if ratio else pd.DataFrame()
Wdata_anchor_scaled   = pd.concat(scaled.values(), axis=1) if scaled else pd.DataFrame()
diag = pd.DataFrame(diag).sort_values('anchor_zero_share', ascending=False)

print(f"Done. {len(raw)} retrieved, {len(failed)} failed: {failed}")

Success for 'average gas prices'.
Success for 'average price of gas'.
Success for 'best gas mileage'.
Success for 'best gas mileage car'.
Success for 'best gas mileage cars'.
Success for 'best gas prices'.
Success for 'best suv gas mileage'.
Success for 'ca gas prices'.
Success for 'california gas price'.
Success for 'california gas prices'.
Success for 'car gas mileage'.
Success for 'cheap gas'.
Success for 'cheap gas prices'.
Success for 'costco gas'.
Success for 'costco gasoline'.
Success for 'current gas price'.
Success for 'gallon of gas price'.
Success for 'gas buddy'.
Success for 'gas calculator'.
Success for 'gas costco price'.
Success for 'gas mileage'.
Success for 'gas mileage calculator'.
Success for 'gas mileage for cars'.
Success for 'gas price average'.
Success for 'gas price news'.
Success for 'gas price per gallon'.
Success for 'gas price today'.
Success for 'gas prices'.
Success for 'gas prices costco'.
Success for 'gas prices in california'.
Success for 'gas prices ne

In [10]:
sum(diag['anchor_zero_share']) # check any infinite ratio happened.

0.0

In [11]:

Wdata_Wgs_weight_only.to_csv(RAW / "gtrends_anchor_ratios.csv")                          # save -> download -> uploaded to github